# 第011讲 线性方程与最小二乘几何

先完成lab.md里的纸笔消元、正交证明与QR手算，再从新内核运行全部单元。数据完全合成、确定、离线。

本讲残差固定为观测减预测。NumPy 2.3.5是核验版本；SVD相关库内部原理在第012讲补充。

## 1 读数据与看形状

第一列全1对应截距，第二列是输入。包装接口要求A二维、y一维，避免不经意广播。

In [1]:
from fractions import Fraction as F
import numpy as np
import experiment as exp

print("NumPy版本：", np.__version__)
print("主例CSV存在：", (exp.BASE / "data" / "main.csv").is_file())
A, y = exp.read_main()
print("A =", A, "形状", A.shape)
print("y =", y, "形状", y.shape)
assert A.shape == (3,2) and y.shape == (3,)

NumPy版本： 2.3.5
主例CSV存在： True
A = [[1. 0.]
 [1. 1.]
 [1. 2.]] 形状 (3, 2)
y = [1. 2. 2.] 形状 (3,)


## 2 精确消元与手算参照

Fraction独立计算正规方程、残差和SSE。精确增广秩检查解释原方程为何无解，而不是把异常当作证明。

In [2]:
exact = exp.exact_main()
print("精确参考：", exact)
print("A 的精确秩：", exp.rank_fraction([[1,0],[1,1],[1,2]]))
print("增广矩阵精确秩：", exp.rank_fraction([[1,0,1],[1,1,2],[1,2,2]]))
assert exact["beta"] == ["7/6", "1/2"]
assert exact["sse"] == "1/6"

精确参考： {'gram': [['3', '3'], ['3', '5']], 'rhs': ['5', '6'], 'beta': ['7/6', '1/2'], 'prediction': ['7/6', '5/3', '13/6'], 'residual': ['-1/6', '1/3', '-1/6'], 'sse': '1/6', 'orthogonality': ['0', '0']}
A 的精确秩： 2
增广矩阵精确秩： 3


## 3 实际方法比较

非方阵不能直接solve或求普通逆。失败状态保留在报告中；它不表示最小二乘不可解。不要预先指定哪条浮点路线每次都最差。

In [3]:
reference = np.array([7/6, 1/2])
main = exp.compare_methods(A, y, reference)
for name, result in main["methods"].items():
    if result["status"] == "ok":
        print(name, "参数", result["beta"], "参数差", result["coefficient_error"], "SSE", result["sse"])
    else:
        print(name, "失败类型", result["error_type"])
assert main["methods"]["solve_A"]["status"] == "failed"

lstsq 参数 [1.1666666666666659, 0.5000000000000001] 参数差 8.95090418262362e-16 SSE 0.16666666666666657
qr_library 参数 [1.1666666666666665, 0.5] 参数差 2.220446049250313e-16 SSE 0.16666666666666669
qr_teaching_mgs 参数 [1.1666666666666674, 0.4999999999999994] 参数差 9.036560719766055e-16 SSE 0.16666666666666666
normal_solve 参数 [1.1666666666666667, 0.5] 参数差 0.0 SSE 0.16666666666666674
normal_inverse 参数 [1.1666666666666663, 0.5] 参数差 4.440892098500626e-16 SSE 0.16666666666666657
solve_A 失败类型 LinAlgError
inverse_A 失败类型 LinAlgError


## 4 QR的每一步都要核验

Q的列符号可能不同，重构、正交与最终求解才是主要标准。这里矩阵缺陷用Frobenius范数，向量误差用欧氏二范数。

In [4]:
Q, R = np.linalg.qr(A, mode="reduced")
Q_hand, R_hand = exp.mgs(A)
print("库 Q：", Q)
print("库 R：", R)
print("教学 Q：", Q_hand)
print("教学 R：", R_hand)
print("库重构缺陷：", np.linalg.norm(Q @ R - A))
print("库正交缺陷：", np.linalg.norm(Q.T @ Q - np.eye(2)))
beta_qr = np.linalg.solve(R, Q.T @ y)
print("QR回代参数：", beta_qr)
assert np.allclose(beta_qr, reference, atol=1e-11, rtol=1e-11)

库 Q： [[-5.77350269e-01  7.07106781e-01]
 [-5.77350269e-01  4.43378006e-17]
 [-5.77350269e-01 -7.07106781e-01]]
库 R： [[-1.73205081 -1.73205081]
 [ 0.         -1.41421356]]
教学 Q： [[ 5.77350269e-01 -7.07106781e-01]
 [ 5.77350269e-01 -1.57009246e-16]
 [ 5.77350269e-01  7.07106781e-01]]
教学 R： [[1.73205081 1.73205081]
 [0.         1.41421356]]
库重构缺陷： 2.2296998761388307e-16
库正交缺陷： 2.482997117251962e-16
QR回代参数： [1.16666667 0.5       ]


## 5 投影不是三维单位变换

P=QQ转置，P对称且重复投影不再改变结果。最优残差不为零，但与每列正交。

In [5]:
P = Q @ Q.T
prediction = P @ y
residual = y - prediction
print("投影矩阵：", P)
print("预测：", prediction)
print("残差：", residual)
print("A转置乘残差：", A.T @ residual)
print("P²-P的矩阵缺陷：", np.linalg.norm(P @ P - P))
assert np.allclose(P, P.T, atol=1e-11, rtol=1e-11)
assert np.allclose(P @ P, P, atol=1e-11, rtol=1e-11)
assert not np.allclose(P, np.eye(3))

投影矩阵： [[ 0.83333333  0.33333333 -0.16666667]
 [ 0.33333333  0.33333333  0.33333333]
 [-0.16666667  0.33333333  0.83333333]]
预测： [1.16666667 1.66666667 2.16666667]
残差： [-0.16666667  0.33333333 -0.16666667]
A转置乘残差： [4.4408921e-16 4.4408921e-16]
P²-P的矩阵缺陷： 1.5700924586837752e-16


## 6 参数族与唯一预测

D复制输入列。比较三组参数只展示现象；最小范数的全体结论仍由讲义的平方展开证明。

In [6]:
D = np.column_stack([A, A[:,1]])
minimum_norm, returned, rank, values = np.linalg.lstsq(D, y, rcond=None)
print("数值秩：", rank, "最小范数系数：", minimum_norm)
print("接口 residuals 返回：", returned)
print("重算SSE：", np.linalg.norm(y-D@minimum_norm)**2)
for t in [-2,0,3]:
    beta = np.array([7/6, 1/4+t, 1/4-t])
    print("t", t, "系数", beta, "长度", np.linalg.norm(beta), "预测", D@beta)
    assert np.allclose(D@beta, A@reference, atol=1e-11, rtol=1e-11)

数值秩： 2 最小范数系数： [1.16666667 0.25       0.25      ]
接口 residuals 返回： []
重算SSE： 0.1666666666666668
t -2 系数 [ 1.16666667 -1.75        2.25      ] 长度 3.079953102096055 预测 [1.16666667 1.66666667 2.16666667]
t 0 系数 [1.16666667 0.25       0.25      ] 长度 1.2190615698606495 预测 [1.16666667 1.66666667 2.16666667]
t 3 系数 [ 1.16666667  3.25       -2.75      ] 长度 4.414307546049676 预测 [1.16666667 1.66666667 2.16666667]


## 7 近共线扫描

每个非零epsilon构造的精确秩都是2。我们同时检查实际存储浮点数的精确有理秩、默认数值秩，以及每个方法的真实误差和失败。

In [7]:
near_results = []
for k in [10,20,26,30,40,50]:
    N, z, true_beta, epsilon, t = exp.near_design(k)
    result = exp.compare_methods(N,z,true_beta)
    near_results.append(result)
    print("k =", k, "epsilon =", epsilon,
          "存储值精确秩", exp.rank_fraction(N.tolist()),
          "默认数值秩", result["default_numeric_rank"],
          "条件数", result["condition_A"])
    for name in ["lstsq", "qr_library", "qr_teaching_mgs", "normal_solve", "normal_inverse"]:
        item = result["methods"][name]
        if item["status"] == "ok":
            print(" ", name, "参数差", item["coefficient_error"], "残差", item["residual_norm"])
        else:
            print(" ", name, item["error_type"], item["message"])

k = 10 epsilon = 0.0009765625 存储值精确秩 2 默认数值秩 2 条件数 1448.1553784042155
  lstsq 参数差 2.1643760136244913e-13 残差 1.4043333874306805e-15
  qr_library 参数差 2.284605923060903e-14 残差 9.930136612989092e-16
  qr_teaching_mgs 参数差 2.983175671499173e-14 残差 0.0
  normal_solve 参数差 0.0 残差 0.0
  normal_inverse 参数差 9.761721496032797e-11 残差 9.76175873397407e-11
k = 20 epsilon = 9.5367431640625e-07 存储值精确秩 2 默认数值秩 2 条件数 1482910.4003391522
  lstsq 参数差 4.0261339236222406e-10 残差 2.8780270159990897e-15
  qr_library 参数差 1.1009103647775406e-10 残差 1.9860273225978185e-15
  qr_teaching_mgs 参数差 9.855336905302195e-11 残差 0.0
  normal_solve 参数差 0.0 残差 0.0
  normal_inverse 参数差 0.00010235906879424087 残差 0.00010235906879461324
k = 26 epsilon = 1.4901161193847656e-08 存储值精确秩 2 默认数值秩 2 条件数 94906265.06767948
  lstsq 参数差 1.4118122386710178e-08 残差 8.881784197001252e-16
  qr_library 参数差 7.987823716277776e-09 残差 0.0
  qr_teaching_mgs 参数差 5.25274275043412e-09 残差 9.930136612989092e-16
  normal_solve 参数差 0.0 残差 0.0
  normal_inverse 参数

## 8 阈值决定有效秩

只改变rcond，不改变数据。高阈值会丢掉弱方向，改变有效问题；不能用更好看的系数反推阈值必然正确。

In [8]:
thresholds = exp.threshold_demo()
for row in thresholds["results"]:
    print(row)
assert [row["reported_rank"] for row in thresholds["results"]] == [2,2,1,1]

{'rcond': None, 'reported_rank': 2, 'beta': [0.9997014805942859, 2.000298519405714], 'residual_norm': 1.2560739669470201e-15, 'coefficient_error': 0.00042217019219231646}
{'rcond': 1e-14, 'reported_rank': 2, 'beta': [0.9997014805942859, 2.000298519405714], 'residual_norm': 1.2560739669470201e-15, 'coefficient_error': 0.00042217019219231646}
{'rcond': 1e-12, 'reported_rank': 1, 'beta': [1.4999999999999998, 1.4999999999999996], 'residual_norm': 1.4380387601476864e-12, 'coefficient_error': 0.7071067811865477}
{'rcond': 1e-10, 'reported_rank': 1, 'beta': [1.4999999999999998, 1.4999999999999996], 'residual_norm': 1.4380387601476864e-12, 'coefficient_error': 0.7071067811865477}


## 9 方向不同的扰动

第一项位于列空间，精确参数变化是(-eta/epsilon,eta/epsilon)；第二项正交于两列，精确预测不变。浮点结果仍要按范围核对。

In [9]:
perturbations = exp.perturbation_demo()
print("精确平行扰动参数变化：", perturbations["exact_parallel_coefficient_change"])
print("正交扰动与各列内积：", perturbations["orthogonal_dot_columns"])
for name, result in perturbations["cases"].items():
    print(name, result)

精确平行扰动参数变化： [-0.0009765625, 0.0009765625]
正交扰动与各列内积： [0.0, 0.0]
column_space {'data_change_norm': 2.9451005721170265e-09, 'coefficient_change_norm': 0.0013810679319661942, 'prediction_change_norm': 2.9451007125503953e-09, 'coefficients': [0.999023437215336, 2.0009765627846625], 'new_residual_norm': 3.1086244689504383e-15}
orthogonal {'data_change_norm': 2.2812650937430355e-09, 'coefficient_change_norm': 1.9785657032707482e-14, 'prediction_change_norm': 9.930136612989092e-16, 'coefficients': [0.9999999997153225, 2.000000000284676], 'new_residual_norm': 2.281265275044931e-09}


## 10 方阵可逆也会敏感

这里用精确分数算epsilon与delta的影响。对角矩阵的条件数则可直接由不同坐标伸缩求出，不需提前使用SVD证明。

In [10]:
epsilon = F(1,1000)
delta = F(1,1000000)
changed = [1-delta/epsilon, 1+delta/epsilon]
print("精确扰动后参数：", [str(x) for x in changed])
T = np.diag([1., 0.01])
print("对角矩阵条件数：", np.linalg.cond(T))
print("正规矩阵条件数：", np.linalg.cond(T.T@T))
assert changed == [F(999,1000),F(1001,1000)]

精确扰动后参数： ['999/1000', '1001/1000']
对角矩阵条件数： 100.0
正规矩阵条件数： 10000.0


## 11 边界与接口返回值

包含形状拒绝、单列、单样本、宽矩阵、零设计、精确秩、QR和行缩放。零设计的空residuals返回并不等于残差为零。

In [11]:
checks, boundary_cases = exp.boundary_tests()
print("边界检查组数：", len(checks))
for item in checks:
    print("通过", item)
zero = boundary_cases["zero"]["methods"]["lstsq"]
print("零设计返回 residuals：", zero["returned_residuals"])
print("零设计重算 SSE：", zero["sse"])
print("行缩放前后解：", boundary_cases["row_scaling"])
assert len(checks) == 14 and zero["sse"] == 14

边界检查组数： 14
通过 one_dimensional_A_rejected
通过 column_y_rejected
通过 row_mismatch_rejected
通过 empty_rows_rejected
通过 empty_columns_rejected
通过 nan_input_rejected
通过 single_equation_single_parameter
通过 single_column_mean
通过 wide_minimum_norm
通过 zero_design_nonzero_residual_despite_empty_return
通过 qr_shapes_orthogonality_reconstruction_projection
通过 exact_augmented_rank_detects_inconsistency
通过 row_scaling_changes_least_squares
通过 diagonal_condition_number_squared
零设计返回 residuals： []
零设计重算 SSE： 14.0
行缩放前后解： {'original': 0.9999999999999998, 'scaled': 1.98019801980198}


## 12 整体重跑并比较

报告JSON与方法CSV包含全部实测值。核验不预设求解器排序，也不把有限案例当作一般稳定性证明。

In [12]:
report = exp.run_experiment(write_outputs=True)
exp.print_report(report)
assert report["exact_main"] == exact
assert report["main"] == main
assert report["threshold_demo"] == thresholds
assert report["perturbations"] == perturbations
assert report["boundary_tests"] == checks
print("脚本与逐步核对一致：", True)
print("报告存在：", (exp.BASE / "outputs" / "experiment_report.json").is_file())

NumPy 2.3.5 ；所有数据为确定性合成数据
精确主例系数： ['7/6', '1/2']
精确残差： ['-1/6', '1/3', '-1/6'] SSE = 1/6
秩亏最小范数系数： [1.166666666666666, 0.2500000000000001, 0.2500000000000001]
近共线实际结果：k / 数值秩 / lstsq参数误差 / normal_solve状态 / normal_inverse状态
10 2 2.164e-13 ok ok
20 2 4.026e-10 ok ok
26 2 1.412e-08 ok ok
30 2 2.259e-07 failed failed
40 2 4.222e-04 failed failed
50 1 7.071e-01 failed failed
阈值实验返回秩： [2, 2, 1, 1]
列空间内扰动的参数变化： 0.0013810679319661942
同次预测变化： 2.9451007125503953e-09
形状与边界检查通过： 14
核验通过；未预设每个例子的求解器误差排名。
脚本与逐步核对一致： True
报告存在： True


## 你的结论

说明主例为何无精确解却有唯一最小二乘参数；说明秩亏时哪一对象唯一；解释一个小残差但参数差不小的案例；记录阈值与环境。

执行边界：制作时用新Python进程中的真实ipykernel进程内内核顺序执行并保留输出。没有测试浏览器Jupyter UI、外进程socket传输或学习者本机Anaconda安装。